# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** Algorithmic Asset Allocation (Hierarchical Risk Parity)

---
*Note: Schumpeterian disruption in capital allocation theory can be encoded using a portfolio optimizer that transcends Markowitz mean-variance. This notebook implements Hierarchical Risk Parity (HRP), applying graph theory and unsupervised hierarchical clustering methods to partition assets according to their correlation topology. The resulting architecture eliminates the algorithmic instability inherent in traditional covariance matrix inversion.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("mako")
np.random.seed(42)

In [ ]:
# Load deterministic dataset
df = pd.read_csv('../data/correlated_assets_synthetic.csv', index_col=0, parse_dates=True)
returns = df.pct_change().dropna()

print(f"Log-returns shape: {returns.shape}")

### 1. Topological Inference (Correlation -> Distance)
The fundamental metric for unsupervised ML in finance transforms cross-correlation into a matrix of verifiable metric distances, satisfying the triangle inequality.

In [ ]:
cov_matrix = returns.cov()
corr_matrix = returns.corr()

# Metric Distance: D_ij = sqrt(0.5 * (1 - rho_ij))
dist_matrix = np.sqrt(0.5 * (1 - corr_matrix).round(10)) # precision round

### 2. Hierarchical Clustering (Ward's Linkage)
We apply the Linkage algorithm on the distance matrix to discover the tree structure (topology) inherent in stock returns.

In [ ]:
# Convert metrics distance to condensed distance list for Scipy linkage
condensed_dist = squareform(dist_matrix, checks=False)

# Hierarchical Clustering algorithm
link_matrix = linkage(condensed_dist, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(link_matrix, labels=df.columns, ax=ax, leaf_rotation=90)
ax.set_title("Asset Topological Structure (HRP Dendrogram)", fontsize=14)
ax.set_ylabel("Ward Distance")
plt.show()

### 3. Quasi-Diagonalization and Recursive Risk Allocation (Recursive Bisection)
Unlike Markowitz, which suffers from instability in the inversion of singular matrices $w = \Sigma^{-1}\mu$, HRP recursively traverses the resulting graph assigning capital (inverse of the cluster-by-cluster variance).

In [ ]:
def get_quasi_diag(link):
    return leaves_list(link)

def get_cluster_var(cov, items):
    cov_ = cov.iloc[items, items]
    ivp = 1.0 / np.diag(cov_) # Inverse Variance Portfolio within cluster
    ivp /= ivp.sum()
    # Variance of the cluster
    w_ = ivp.reshape(-1, 1)
    cluster_var = np.dot(np.dot(w_.T, cov_), w_)[0, 0]
    return cluster_var

def get_hrp_weights(cov, sort_ix):
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix] # Initialize with all sorted items
    
    while len(c_items) > 0:
        # Bisect
        c_items = [i[int(j):int(k)] 
                   for i in c_items 
                   for j, k in ((0, len(i)/2), (len(i)/2, len(i))) if len(i) > 1]
        
        for i in range(0, len(c_items), 2):
            c_items0 = c_items[i]    # Left cluster
            c_items1 = c_items[i+1]  # Right cluster
            
            c_var0 = get_cluster_var(cov, c_items0)
            c_var1 = get_cluster_var(cov, c_items1)
            
            alpha = 1 - c_var0 / (c_var0 + c_var1)
            
            w[c_items0] *= alpha # Weight for left cluster
            w[c_items1] *= (1 - alpha) # Weight for right cluster
            
    return w

ordered_indices = get_quasi_diag(link_matrix)
hrp_weights = get_hrp_weights(cov_matrix, ordered_indices)

# Reorder back to original asset names to print results
hrp_weights.index = returns.columns[hrp_weights.index]
hrp_weights.sort_values(ascending=False, inplace=True)

print("Top 10 HRP Capital Allocation Weights:")
print(hrp_weights.head(10).round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(x=hrp_weights.index, y=hrp_weights.values, ax=ax, color="#4C72B0")
ax.set_title("Vectorial Capital Allocation (Hierarchical Risk Parity)", fontsize=14)
ax.set_ylabel("Weight")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
plt.tight_layout()
plt.show()